# B → Xᵤℓν hybrid model — example

This notebook walks through computing both the optimal transport (OT) hybrid weights
and the conventional bin-by-bin hybrid weights for a single decay mode.

Set the paths in the **Configuration** cell below, then run all cells in order.

In [ ]:
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt

from hybrid import (
    build_phase_space_grid,
    solve_ot,
    extract_reweights,
    apply_ot_weights,
    plot_sink_distribution,
    compute_normalization,
    compute_conventional_weights,
    apply_bin_weights,
    compute_all_moments,
    plot_moment_comparison,
    print_moment_errors,
)


## Configuration

All parameters are read from `config.yaml` in the same directory. Edit that file to set your file paths, branching fractions, and OT settings before running the cells below.


In [ ]:
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

PATH_INCLUSIVE = cfg["input"]["inclusive"]
PATH_RESONANT  = cfg["input"]["resonant"]
MODE       = cfg["mode"]
INCL_MODEL = cfg.get("inclusive_model", "inclusive")
BFS        = {int(k): v for k, v in cfg["branching_fractions"][MODE].items()}
INCL_BF    = cfg["inclusive_branching_fraction"]
BIN_WIDTH  = cfg["optimal_transport"]["bin_width"]
PMINUS_RANGE = tuple(cfg["optimal_transport"]["pminus_range"])
PPLUS_RANGE  = tuple(cfg["optimal_transport"]["pplus_range"])
SIGNAL_COL   = cfg["input"].get("signal_filter_col")
BINNING = {k: np.array(v) for k, v in cfg["conventional_hybrid"]["binning"].items()}


## 1. Load data

In [ ]:
columns = [
    cfg["input"].get("pdg_col",    "X_gen_PDG"),
    cfg["input"].get("pplus_col",  "genPplus"),
    cfg["input"].get("pminus_col", "genPminus"),
    "genMx", "genq2", "gen_lep_E_B", "genCosThetaL",
]
if SIGNAL_COL:
    columns.append(SIGNAL_COL)

def read_file(path, cols):
    if path.endswith(".pq") or path.endswith(".parquet"):
        df = pd.read_parquet(path, columns=cols)
    else:
        with uproot.open(path) as f:
            tree_name = next(k.split(";")[0] for k in f.keys())
            df = f[tree_name].arrays(cols, library="pd")
    if SIGNAL_COL:
        df = df[df[SIGNAL_COL] > 0].reset_index(drop=True)
    return df

df_incl = read_file(PATH_INCLUSIVE, columns)
df_excl = read_file(PATH_RESONANT,  columns)

print(f"Inclusive events: {len(df_incl):,}")
print(f"Resonant events:  {len(df_excl):,}")


## 2. Normalization weights

The exclusive sample needs to be scaled so that the combined hybrid model integrates to the
correct total inclusive branching fraction. The normalization factor is:

$$w_\mathrm{excl} = \frac{N_\mathrm{incl}}{N_\mathrm{excl}} \left(1 - \frac{\mathcal{B}_{X_u}}{\sum_i \mathcal{B}_i + \mathcal{B}_{X_u}}\right)$$

This adds a `norm_weight` column to both DataFrames.

In [ ]:
compute_normalization(
    df_incl, df_excl,
    branching_fractions=BFS,
    inclusive_bf=INCL_BF,
)
print("excl normalization weight:", df_excl["norm_weight"].iloc[0])

## 3. OT hybrid weights

We build 2D histograms in (P+, P-) space for both samples, solve the optimal transport
problem to find the minimal-cost redistribution, and extract per-bin reweights.

In [ ]:
Pinc, Pres = build_phase_space_grid(
    df_incl, df_excl,
    bin_width=BIN_WIDTH,
    pminus_range=PMINUS_RANGE,
    pplus_range=PPLUS_RANGE,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, dist, title in zip(axes, [Pinc, Pres], ["Inclusive", "Resonant"]):
    im = ax.imshow(dist, origin="lower", aspect="auto", cmap="magma",
                   extent=[PMINUS_RANGE[0], PMINUS_RANGE[1], PPLUS_RANGE[0], PPLUS_RANGE[1]])
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(r"$P^-$ [GeV]")
    ax.set_ylabel(r"$P^+$ [GeV]")
    plt.colorbar(im, ax=ax)
plt.tight_layout()


In [ ]:
G, src_coords, src_shape = solve_ot(
    Pinc, Pres,
    bin_width=BIN_WIDTH,
    window=cfg["optimal_transport"].get("window"),
    lambda_reg=cfg["optimal_transport"].get("lambda_reg", 0),
    verbose=True,
)

reweight_map = extract_reweights(G, src_coords, src_shape, Pinc)
df_incl["ot_hybrid_weight"] = apply_ot_weights(df_incl, reweight_map, BIN_WIDTH)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(reweight_map, origin="lower", aspect="auto", cmap="viridis",
               extent=[PMINUS_RANGE[0], PMINUS_RANGE[1], PPLUS_RANGE[0], PPLUS_RANGE[1]],
               vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Weight")
ax.set_xlabel(r"$P^-$ [GeV]")
ax.set_ylabel(r"$P^+$ [GeV]")
ax.set_title("OT hybrid reweights")
plt.tight_layout()


## 4. Conventional bin-by-bin hybrid (for comparison)

In [ ]:
conv_arr = compute_conventional_weights(df_incl, df_excl, BINNING, weight_col="norm_weight")
df_incl["conventional_hybrid_weight"] = apply_bin_weights(df_incl, BINNING, conv_arr)


## 5. Kinematic distributions

Compare inclusive DFN vs. OT hybrid vs. conventional hybrid for the four main variables.

In [ ]:
from plothist import get_color_palette

variables = [
    ("genMx",        np.linspace(0, 3.51, 50),  r"$M_X$ [GeV]"),
    ("gen_lep_E_B",  np.linspace(0, 2.65, 50),  r"$E_\ell^B$ [GeV]"),
    ("genq2",        np.linspace(0, 25.01, 50), r"$q^2$ [GeV$^2$]"),
    ("genCosThetaL", np.linspace(-1, 1, 50),    r"$\cos\theta_\ell$"),
]

if MODE == "charged":
    masks = {
        r"$\pi^0$":  df_excl["X_gen_PDG"].abs() == 111,
        r"$\eta$":   df_excl["X_gen_PDG"].abs() == 221,
        r"$\rho^0$": df_excl["X_gen_PDG"].abs() == 113,
        r"$\omega$": df_excl["X_gen_PDG"].abs() == 223,
        r"$\eta'$":  df_excl["X_gen_PDG"].abs() == 331,
    }
else:
    masks = {
        r"$\pi^\pm$": df_excl["X_gen_PDG"].abs() == 211,
        r"$\rho^\pm$":df_excl["X_gen_PDG"].abs() == 213,
    }

palette = get_color_palette("cubehelix", len(masks) + 1)[::-1]
def rgba(k):
    r, g, b = palette[k][:3]
    return (r, g, b, 0.85)
all_labels = ["Incl leftover"] + list(masks.keys())
all_colors = [rgba(0)] + [rgba(j + 1) for j in range(len(masks))]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
mode_label = r"$B^\pm$" if MODE == "charged" else r"$B^0$"

for i, (var, bins, xlabel) in enumerate(variables):
    ax = axes[i]

    h_dfn  = np.histogram(df_incl[var], bins=bins, weights=df_incl["norm_weight"])[0]
    h_incl = np.histogram(df_incl[var], bins=bins,
                           weights=df_incl["norm_weight"] * df_incl["ot_hybrid_weight"])[0]
    h_excl = [np.histogram(df_excl[var][m], bins=bins,
                            weights=df_excl["norm_weight"][m])[0]
              for m in masks.values()]

    bottom = np.zeros_like(h_incl, dtype=float)
    for hdat, lab, col in zip([h_incl] + h_excl, all_labels, all_colors):
        ax.bar(bins[:-1], hdat, width=np.diff(bins), bottom=bottom,
               align="edge", color=col, edgecolor="none", label=lab)
        bottom += hdat

    ax.step(bins[:-1], bottom, where="post", color="k", lw=1.5, label="OT hybrid")
    ax.step(bins[:-1], h_dfn,  where="post", color="r", lw=1.2, label=INCL_MODEL)
    ax.text(0.05, 0.95, mode_label, transform=ax.transAxes, va="top", ha="left", fontsize=14)
    if i == 0:
        ax.legend(fontsize=11)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel("Events", fontsize=13)
    ax.grid(alpha=0.3)

plt.tight_layout()

## 6. Moment comparison

How well does each method preserve the HQE spectral moments?

In [ ]:
for df in (df_incl, df_excl):
    df["genMxSq"] = df["genMx"] ** 2
df_excl["_null"] = 0.0
df_incl["_conv_wt"] = df_incl["norm_weight"] * df_incl["conventional_hybrid_weight"]

varlist = ["genMxSq", "genq2", "gen_lep_E_B"]
kw = dict(df_incl=df_incl, df_excl=df_excl, varlist=varlist, n=4)

mom_ref      = compute_all_moments(**kw, incl_weight_col="norm_weight",  excl_weight_col="_null",      central=False)
mom_ref_cen  = compute_all_moments(**kw, incl_weight_col="norm_weight",  excl_weight_col="_null",      central=True)
mom_ot       = compute_all_moments(**kw, incl_weight_col="ot_hybrid_weight",   excl_weight_col="norm_weight", central=False)
mom_ot_cen   = compute_all_moments(**kw, incl_weight_col="ot_hybrid_weight",   excl_weight_col="norm_weight", central=True)
mom_conv     = compute_all_moments(**kw, incl_weight_col="_conv_wt",      excl_weight_col="norm_weight", central=False)
mom_conv_cen = compute_all_moments(**kw, incl_weight_col="_conv_wt",      excl_weight_col="norm_weight", central=True)

print_moment_errors(mom_ot, mom_conv, mom_ref, mom_ot_cen, mom_conv_cen, mom_ref_cen, varlist)

In [ ]:
fig = plot_moment_comparison(
    mom_ot, mom_conv, mom_ref,
    mom_ot_cen, mom_conv_cen, mom_ref_cen,
    varlist, MODE,
)